In [61]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.mixed_precision import Policy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras import regularizers
import warnings

In [62]:
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [63]:
# Load data
df = pd.read_csv("data/train.csv")
image_dir = "data/train_images/train_images/"

# Split data into training and validation sets for now, later use the test set for final evaluation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [64]:
le = LabelEncoder()
train_labels = le.fit_transform(train_df['TARGET'])
val_labels = le.transform(val_df['TARGET'])

train_paths = image_dir + train_df['file_name'].astype(str)
val_paths = image_dir + val_df['file_name'].astype(str)

def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    label = tf.one_hot(label, depth=100) 
    return img, label

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 512

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = (train_ds
    .shuffle(buffer_size=len(train_paths))
    .map(process_path, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = (val_ds
    .map(process_path, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [65]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [70]:
data_augmentation = tf.keras.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.11), # 0.11 is roughly 40 degrees (40/360)
    layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
    layers.RandomZoom(height_factor=0.2, width_factor=0.2),
], name="data_augmentation")

model_CNN = models.Sequential([
    data_augmentation,
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(100, activation='softmax')
])

model_CNN.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model_CNN.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_27 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_28 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 512)            │    44,302,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 100)            │        51,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,447,396 (169.55 MB)

 Trainable params: 44,447,396 (169.55 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
history = model_CNN.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.0112 - loss: 6.6852 - val_accuracy: 0.0091 - val_loss: 4.5995
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step - accuracy: 0.0325 - loss: 4.4412 - val_accuracy: 0.0699 - val_loss: 4.0711
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step - accuracy: 0.1158 - loss: 3.7332 - val_accuracy: 0.1965 - val_loss: 3.4479
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.2042 - loss: 3.2130 - val_accuracy: 0.2513 - val_loss: 3.1726
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step - accuracy: 0.2807 - loss: 2.8286 - val_accuracy: 0.3577 - val_loss: 2.6164
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 128ms/step - accuracy: 0.3445 - loss: 2.5714 - val_accuracy: 0.3867 - val_loss: 2.4450
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 128ms/step - accuracy: 0.3821 - loss: 2.3820 - val_accuracy: 0.4343 - val_loss: 2.2578
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.4069 - loss: 2.2766 - val_accura